In [1]:
import cv2
import numpy as np
from PIL import Image
import requests
from io import BytesIO

IMAGE_URL = "http://localhost:8080/2025-02-01T1458_NB_generated_002.jpg"
STRAIGHTENED_PATH = "/tmp/straightened.jpg"

response = requests.get(IMAGE_URL)
img = np.array(Image.open(BytesIO(response.content)).convert("RGB"))
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
edges = cv2.Canny(binary, 50, 150, apertureSize=3)

# 0.1° resolution to catch sub-degree skew
lines = cv2.HoughLines(edges, 1, np.pi / 1800, threshold=100)

angles_vertical = []
if lines is not None:
    for line in lines:
        rho, theta = line[0]
        angle = np.degrees(theta) - 90
        if abs(angle) < 5:
            angles_vertical.append(angle)

skew_angle = float(np.median(angles_vertical)) if angles_vertical else 0.0
print(f"Correction angle: {skew_angle:.2f}°")

h, w = img.shape[:2]
M = cv2.getRotationMatrix2D((w // 2, h // 2), skew_angle, 1.0)
straightened = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC,
                               borderMode=cv2.BORDER_CONSTANT,
                               borderValue=(255, 255, 255))

Image.fromarray(straightened).save(STRAIGHTENED_PATH)
print(f"Saved to {STRAIGHTENED_PATH}")

Correction angle: -0.10°
Saved to /tmp/straightened.jpg


In [1]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_PATH = "zai-org/GLM-OCR"
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant for recognizing text in images. All the text in the image is in Norwegian nynorsk. Extract all text faithfully."
    },
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": "http://localhost:8080/2025-02-01T1458_NB_generated_002.jpg"
            },
            {
                "type": "text",
                "text": "Text Recognition:"
            }
        ],
    }
]
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
inputs.pop("token_type_ids", None)
generated_ids = model.generate(**inputs, max_new_tokens=8192, repetition_penalty=1.3)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)


/home/lars/miniconda3/envs/vllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The image processor of type `Glm46VImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 510/510 [00:00<00:00, 2237.54it/s]


TypeError: string indices must be integers, not 'str'

In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_PATH = "zai-org/GLM-OCR"
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are a helpful assistant for recognizing text in images. Extract all text faithfully."
            },
            {
                "type": "image",
                 "url": "http://localhost:8080/2025-02-01T1458_NB_generated_001.jpg"
            },
            {
                "type": "text",
                "text": "Text Recognition:"
            }
        ],
    }
]
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
inputs.pop("token_type_ids", None)
generated_ids = model.generate(**inputs, max_new_tokens=8192, repetition_penalty=1.3)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)

The image processor of type `Glm46VImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

SOGNS AVIS

NORGS BONDELAGS ORGAN FOR SOGN

TORSDAG 8. JANUAR 1948.

23. årgang

Edgar Henriksen : Flukten fra landsbygda.
Intervjuer og betraktninger.
Aschehoug.

Dette er den mest aktuelle boka eg har lese i år. Og så er ho så godt skrivi. Det författaren har på hjarta, er lagtGod til rette, han vert ädri keidsam. Han er sjölv bonde og veit kvar skoen tek. Han velt mykje mein det. Han serdet som a'le no burde sja, at skal «flUKTen fra Landsbygda» halda fram, kan det vera utе med oss som fok'. Det er landspolitikk. Byane og bystroka ville snart bli avfo'la om dei ikke fekk tilsig frä byugden; men skal del gå på skakke som no det gjer, lazer vel bonderne og snart à setJA ned barnetøret.

Fyrst i boka kjem nokre intervju forffatteren har havrt måndbonder, «sifets folk», og sa eitt avsvitt som heiter 《Bondeungdommen har ordet》。Deir alle misnogde med slitet på jorda'sume har alt sagt gardasarbe'd farvel og er komme på ein fabrikk, eller deier hamna på ei kontor eller i ein butikk i byen. 

In [3]:

# --- Fine-tuned model (5,000 samples, 3 epochs) ---
# Compare output against the base model cell above on the same image.

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Free the base model from GPU memory before loading the fine-tuned one
try:
    del model, processor, inputs, generated_ids
    torch.cuda.empty_cache()
    print("Cleared base model from GPU memory.")
except NameError:
    pass

FINETUNED_PATH = "/mnt/data/development/GLM-OCR-Norwegian-Nynorsk/LLaMA-Factory/saves/glm-ocr-nynorsk/full/sft"

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are a helpful assistant for recognizing text in images. Extract all text faithfully."
            },
            {
                "type": "image",
                "url": "http://localhost:8080/2025-02-01T1458_NB_generated_001.jpg"
            },
            {
                "type": "text",
                "text": "Text Recognition:"
            }
        ],
    }
]

processor = AutoProcessor.from_pretrained(FINETUNED_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    FINETUNED_PATH,
    torch_dtype="auto",
    device_map="auto",
)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
inputs.pop("token_type_ids", None)

generated_ids = model.generate(**inputs, max_new_tokens=8192, repetition_penalty=1.3)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)


Cleared base model from GPU memory.


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

Norges Bondeplags Organ For SoGN

TorSdag 8. januar 1948.

23. årgang

Edgar Henriksen : Flukten fra landsbygda. Intervjuer og betraktninger. Aschehoug.

Dette er den mest aktuelle boka eg har lese i år. Øg så er ho så godt skrivi. Det førfattaren har på hjarta, er lagt godd til rette, han vert å'drì keidssam. Han er sjölv bonde og veit kvart skoen tek. Han velt mykje meir enn det. Han ser det som 'åle no burde sjà', at skal «flukten fra landsbygda» halda fram, kan det vera ute med oss som fø'k'. Det er landspolitikk. Byane og bystro-ka ville snart bli avfol'ka om dei ikkje fekk tilsig frå bygrese; men skal det gå på sakakke som no det gjjer, lærer vel øbønderne og snart å setja ned barnetaletet. Fyrst i boka kjem nokre intervju forfallarten har havt med bondør, ''sjètets folk''), og så eitt avsnitti som heiterer («Bondeungdommen har ordret». Dei er alle misngode med slitet på jorda. Same har alt sagt gardasarbe'det farvel og er komme på ein verk-stad el'e r på ein fabrikk, eller deira